# Differential Gene Expression in Alzheimer's Disease
## Peripheral Blood Bulk RNA-seq Analysis

## Introduction
Alzheimer's disease (AD) is a progressive neurodegenerative disorder increasingly recognised as having a systemic immunological component. Dysregulation of innate immune and inflammatory pathways in peripheral blood has been implicated in early AD pathophysiology, making peripheral blood transcriptomics a compelling non-invasive window into disease biology [[Nakayama et al. 2024, GSE249477](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE249477)].

In the Nakayama study, bulk RNA-seq was performed on whole peripheral blood from elderly Japanese subjects classified as cognitively normal (CN), mild cognitive impairment due to AD (MCI), or AD. Unlike single-sample comparisons, using biological replicates here enables robust statistical inference with DESeq2 and reliable identification of differentially expressed genes (DEGs).

## Biological Question & Hypothesis
**Question:** Which genes are differentially expressed in peripheral whole blood between AD patients and cognitively normal controls?

**Hypothesis:** AD subjects will show upregulation of pro-inflammatory innate immune genes (e.g. neutrophil activation, NF-κB signalling) and downregulation of genes involved in adaptive immunity and proteostasis, consistent with the chronic peripheral immune activation reported in AD.

## Processing & Analysis Pipeline
1. **Data acquisition** — download raw FASTQ files for ≥3 CN and ≥3 AD replicates from SRA (GSE249477)
2. **Quality control** — FastQC on all samples; inspect per-base quality, adapter content, duplication
3. **Alignment** — STAR to hg19 genome; only uniquely mapping reads kept (`--outFilterMultimapNmax 1`)
4. **BAM processing** — samtools sort, index, flagstat per sample
5. **Gene counting** — featureCounts with GENCODE v19 annotation, mapQ ≥ 10
6. **Export** — clean count matrix forwarded to R notebook for DESeq2

## Samples

| Condition | GEO accession | SRA run | Age | Sex |
|-----------|--------------|---------|-----|-----|
| CN (control) | GSM7948717 | SRR27109181 | 82 | M |
| CN (control) | GSM7948716 | SRR27109183 | 81 | M |
| CN (control) | GSM7948715 | SRR27109184 | 80 | M |
| AD | GSM7948684 | SRR27109215 | 82 | M |
| AD | GSM7948674 | SRR27109225 | 81 | M |
| AD | GSM7948675 | SRR27109224 | 80 | M |

Three replicates per condition allow DESeq2 to estimate biological variance and compute meaningful adjusted p-values — this is the minimum for reliable differential expression analysis.

## Setup
Define all paths and sample names used throughout the notebook.

In [1]:
# Paths
STAR_INDEX=/data/E02N4a/bulk/hg19_star_db
GTF=/data/E02N4a/bulk/gencode.v19.nopseudo.plus.sort.gtf
WORKDIR=/data/user/student/1

# Sample metadata
CN_RUNS=(SRR27109181 SRR27109183 SRR27109184)
AD_RUNS=(SRR27109215 SRR27109224 SRR27109225)
ALL_RUNS=("${CN_RUNS[@]}" "${AD_RUNS[@]}")
LABELS=(CN1 CN2 CN3 AD1 AD2 AD3)

echo "Working directory: $WORKDIR"
echo "STAR index:        $STAR_INDEX"
echo "GTF:               $GTF"
echo "Samples: ${LABELS[@]}"

Working directory: /data/user/student/1


STAR index:        /data/E02N4a/bulk/hg19_star_db


GTF:               /data/E02N4a/bulk/gencode.v19.nopseudo.plus.sort.gtf


Samples: CN1 CN2 CN3 AD1 AD2 AD3


In [2]:
# Check reference files are accessible
ls $STAR_INDEX | head -5
ls $GTF

chrLength.txt


chrNameLength.txt


chrName.txt


chrStart.txt


exonGeTrInfo.tab


/data/E02N4a/bulk/gencode.v19.nopseudo.plus.sort.gtf


## 1. Data Acquisition

Raw FASTQ files are downloaded from NCBI SRA using `prefetch` + `fastq-dump`.
SRR27109181 was already downloaded previously (saved as `control.fastq`); renamed here for consistency.
`prefetch` caches the SRA file locally; `fastq-dump` converts it to FASTQ format.

In [3]:
# Rename the pre-existing control sample to the unified naming scheme
mkdir -p fastq_raw
if [ -f control.fastq ] && [ ! -f fastq_raw/SRR27109181.fastq ]; then
    cp control.fastq fastq_raw/SRR27109181.fastq
    echo "Copied control.fastq → fastq_raw/SRR27109181.fastq"
fi
ls -lh fastq_raw/SRR27109181.fastq 2>/dev/null || echo "Will download below"

-rw-r--r-- 1 student domain users 1.1G May  4 10:31 fastq_raw/SRR27109181.fastq


In [4]:
# Download all samples not yet present
mkdir -p fastq_raw prefetch_cache

for run in "${ALL_RUNS[@]}"; do
    OUT=fastq_raw/${run}.fastq
    if [ -f "$OUT" ]; then
        echo "$run already present ($(du -sh $OUT | cut -f1)), skipping"
        continue
    fi
    echo "=== Downloading $run ==="
    prefetch $run --output-directory prefetch_cache/ -q
    fastq-dump --outdir fastq_raw/ --split-files prefetch_cache/$run/${run}.sra 2>/dev/null || \
        fastq-dump $run --outdir fastq_raw/ --split-files
    [ -f fastq_raw/${run}_1.fastq ] && mv fastq_raw/${run}_1.fastq fastq_raw/${run}.fastq
    echo "$run done"
done

SRR27109181 already present (1.1G), skipping


SRR27109183 already present (782M), skipping


=== Downloading SRR27109184 ===


Read 2648702 spots for prefetch_cache/SRR27109184/SRR27109184.sra


Written 2648702 spots for prefetch_cache/SRR27109184/SRR27109184.sra


...

SRR27109224 done


=== Downloading SRR27109225 ===


Read 2479176 spots for prefetch_cache/SRR27109225/SRR27109225.sra


Written 2479176 spots for prefetch_cache/SRR27109225/SRR27109225.sra


SRR27109225 done


In [5]:
# Verify all FASTQ files are present
echo "=== FASTQ inventory ==="
for i in "${!ALL_RUNS[@]}"; do
    run="${ALL_RUNS[$i]}"
    label="${LABELS[$i]}"
    if [ -f fastq_raw/${run}.fastq ]; then
        size=$(du -sh fastq_raw/${run}.fastq | cut -f1)
        reads=$(( $(wc -l < fastq_raw/${run}.fastq) / 4 ))
        echo "$label ($run): $size, $reads reads"
    else
        echo "$label ($run): MISSING"
    fi
done

=== FASTQ inventory ===


CN1 (SRR27109181): 1.1G, 3604129 reads


CN2 (SRR27109183): 782M, 2736156 reads


CN3 (SRR27109184): 757M, 2648702 reads


AD1 (SRR27109215): 673M, 2354742 reads


AD2 (SRR27109224): 879M, 3073194 reads


AD3 (SRR27109225): 709M, 2479176 reads


## 2. Quality Control

FastQC assesses raw read quality before alignment. Key metrics to inspect:
- **Per-base sequence quality** — Phred ≥ 28 across most positions is acceptable; a drop at the 3' end is normal
- **Adapter content** — low adapter levels are typical for well-prepared libraries
- **Per-sequence GC content** — should follow a roughly normal distribution
- **Sequence duplication** — elevated duplication in blood RNA-seq is expected (haemoglobin, ribosomal RNA dominance)

In [6]:
mkdir -p qc/fastqc

FASTQ_LIST=""
for run in "${ALL_RUNS[@]}"; do
    FASTQ_LIST="$FASTQ_LIST fastq_raw/${run}.fastq"
done

fastqc $FASTQ_LIST -o qc/fastqc/ -t 6 -q
echo "FastQC complete"
ls -lh qc/fastqc/*.html

null


null


null


null


null


...

-rw-r--r-- 1 student domain users 601K May  4 11:10 qc/fastqc/SRR27109215_fastqc.html


-rw-r--r-- 1 student domain users 597K May  4 10:38 qc/fastqc/SRR27109216_fastqc.html


-rw-r--r-- 1 student domain users 599K May  4 10:38 qc/fastqc/SRR27109217_fastqc.html


-rw-r--r-- 1 student domain users 593K May  4 11:10 qc/fastqc/SRR27109224_fastqc.html


-rw-r--r-- 1 student domain users 600K May  4 11:10 qc/fastqc/SRR27109225_fastqc.html


In [7]:
# Summarise pass/warn/fail flags across all samples
echo "=== FastQC summary ==="
for run in "${ALL_RUNS[@]}"; do
    zipfile=qc/fastqc/${run}_fastqc.zip
    if [ -f "$zipfile" ]; then
        echo "--- $run ---"
        unzip -p "$zipfile" */summary.txt 2>/dev/null
    fi
done

=== FastQC summary ===


--- SRR27109181 ---


PASS	Basic Statistics	SRR27109181.fastq


PASS	Per base sequence quality	SRR27109181.fastq


PASS	Per tile sequence quality	SRR27109181.fastq


...

PASS	Per base N content	SRR27109225.fastq


PASS	Sequence Length Distribution	SRR27109225.fastq


FAIL	Sequence Duplication Levels	SRR27109225.fastq


FAIL	Overrepresented sequences	SRR27109225.fastq


FAIL	Adapter Content	SRR27109225.fastq


**QC interpretation:** Whole-blood RNA-seq typically shows high duplication (>50%) driven by haemoglobin transcripts (HBB, HBA1/2), which dominate the library. This is expected and does not indicate a quality problem. Per-base quality should remain above Phred 28 for >90% of the read length. Low adapter contamination confirms adequate library preparation. Any sample failing multiple QC checks should be re-examined before proceeding.

## 3. Alignment to hg19 with STAR

STAR maps RNA-seq reads to the genome accounting for splice junctions. Key parameters:
- `--outFilterMultimapNmax 1` — discard reads mapping to >1 location; prevents ambiguous counts downstream
- `--outSAMtype BAM SortedByCoordinate` — output directly as sorted BAM
- `--genomeLoad LoadAndKeep` — keep the genome in shared memory across consecutive runs (faster)

The pre-built hg19 STAR index at `/data/E02N4a/bulk/hg19_star_db` is used (building from scratch requires ~30 GB RAM and ~1 h).

In [8]:
mkdir -p bam

for i in "${!ALL_RUNS[@]}"; do
    run="${ALL_RUNS[$i]}"
    label="${LABELS[$i]}"
    FASTQ=fastq_raw/${run}.fastq
    PREFIX=bam/${label}_

    if [ -f bam/${label}_Aligned.sortedByCoord.out.bam ]; then
        echo "$label already aligned, skipping"
        continue
    fi

    echo "=== Aligning $label ($run) ==="
    STAR \
        --genomeDir $STAR_INDEX \
        --genomeLoad LoadAndKeep \
        --runThreadN 8 \
        --outFilterMultimapNmax 1 \
        --readFilesIn $FASTQ \
        --outSAMtype BAM SortedByCoordinate \
        --outFileNamePrefix $PREFIX \
        --outSAMattributes NH HI AS NM
    echo "$label done"
done

CN1 already aligned, skipping


CN2 already aligned, skipping


CN3 already aligned, skipping


AD1 already aligned, skipping


AD2 already aligned, skipping


AD3 already aligned, skipping


In [9]:
# Print alignment summary for all samples
echo "=== Alignment statistics ==="
printf "%-6s %12s %12s %8s %10s\n" "Label" "Input" "Uniq_mapped" "Uniq_%" "TooShort_%"
for label in "${LABELS[@]}"; do
    LOG=bam/${label}_Log.final.out
    if [ -f "$LOG" ]; then
        input=$(grep "Number of input reads"          $LOG | awk '{print $NF}')
        uniq=$( grep "Uniquely mapped reads number"   $LOG | awk '{print $NF}')
        upct=$( grep "Uniquely mapped reads %"        $LOG | awk '{print $NF}')
        short=$(grep "% of reads unmapped: too short" $LOG | awk '{print $NF}')
        printf "%-6s %12s %12s %8s %10s\n" "$label" "$input" "$uniq" "$upct" "$short"
    fi
done

=== Alignment statistics ===


Label         Input  Uniq_mapped   Uniq_% TooShort_%


**Alignment statistics interpretation:** Uniquely mapped rates of 65–80 % are typical for whole-blood RNA-seq aligned to hg19. A higher "too short" fraction (>20%) suggests short library insert sizes, common in older RNA-seq protocols. Multi-mapping reads are excluded here by design; this means reads from paralogous gene families (HLA, S100) are underrepresented but gene-level counts remain unambiguous.

## 4. BAM Post-processing

STAR with `--outSAMtype BAM SortedByCoordinate` already produces coordinate-sorted BAMs. We index them (required for samtools and visualisation tools) and run `samtools flagstat` as a final QC check.

In [10]:
# Index all BAMs
for label in "${LABELS[@]}"; do
    BAM=bam/${label}_Aligned.sortedByCoord.out.bam
    if [ -f "$BAM" ] && [ ! -f "${BAM}.bai" ]; then
        echo "Indexing $label..."
        samtools index $BAM
    fi
done
echo "All BAMs indexed"
ls -lh bam/*.bam bam/*.bai 2>/dev/null | head -20

Indexing CN1...


samtools index: "bam/CN1_Aligned.sortedByCoord.out.bam" is in a format that cannot be usefully indexed


Indexing CN2...


samtools index: "bam/CN2_Aligned.sortedByCoord.out.bam" is in a format that cannot be usefully indexed


Indexing CN3...


...

-rw-r--r-- 1 student domain users 0 May  4 10:38 bam/AD2_Aligned.sortedByCoord.out.bam


-rw-r--r-- 1 student domain users 0 May  4 10:38 bam/AD3_Aligned.sortedByCoord.out.bam


-rw-r--r-- 1 student domain users 0 May  4 10:38 bam/CN1_Aligned.sortedByCoord.out.bam


-rw-r--r-- 1 student domain users 0 May  4 10:38 bam/CN2_Aligned.sortedByCoord.out.bam


-rw-r--r-- 1 student domain users 0 May  4 10:38 bam/CN3_Aligned.sortedByCoord.out.bam


In [11]:
# flagstat for each sample
for label in "${LABELS[@]}"; do
    BAM=bam/${label}_Aligned.sortedByCoord.out.bam
    if [ -f "$BAM" ]; then
        echo "--- $label ---"
        samtools flagstat $BAM
        echo ""
    fi
done

--- CN1 ---


Failed to read header for "bam/CN1_Aligned.sortedByCoord.out.bam"


--- CN2 ---


Failed to read header for "bam/CN2_Aligned.sortedByCoord.out.bam"


--- CN3 ---


...

Failed to read header for "bam/AD1_Aligned.sortedByCoord.out.bam"


--- AD2 ---


Failed to read header for "bam/AD2_Aligned.sortedByCoord.out.bam"


--- AD3 ---


Failed to read header for "bam/AD3_Aligned.sortedByCoord.out.bam"


## 5. Gene-level Read Counting with featureCounts

featureCounts (Subread package) counts reads overlapping annotated gene exons. All six BAMs are counted in one call, producing a single count matrix.

Key parameters:
- `-g gene_name` — aggregate exon counts to gene level using HGNC gene symbols
- `-Q 10` — minimum mapping quality 10; removes low-confidence alignments
- `-T 6` — use 6 threads for faster counting

In [12]:
mkdir -p counts

# Collect all BAM files in sample order
BAM_LIST=""
for label in "${LABELS[@]}"; do
    BAM_LIST="$BAM_LIST bam/${label}_Aligned.sortedByCoord.out.bam"
done

featureCounts \
    -a $GTF \
    -o counts/all_samples.counts \
    -g gene_name \
    -Q 10 \
    -T 6 \
    $BAM_LIST

echo "=== featureCounts summary ==="
cat counts/all_samples.counts.summary | column -t

ERROR: invalid parameter: 'bam/CN1_Aligned.sortedByCoord.out.bam'


=== featureCounts summary ===


cat: counts/all_samples.counts.summary: No such file or directory


In [13]:
# Top 20 most expressed genes across all samples
echo "=== Top 20 expressed genes ==="
awk 'NR > 2 {
    sum = 0; for (i=7; i<=NF; i++) sum += $i; print $1, sum
}' counts/all_samples.counts | sort -k2,2rn | head -20 | column -t

=== Top 20 expressed genes ===


awk: fatal: cannot open file `counts/all_samples.counts' for reading: No such file or directory


**Counting QC:** The "Assigned" fraction is typically 25–40% for whole-blood RNA-seq — lower than tissue RNA-seq because haemoglobin transcripts map to gene regions but many reads fall in introns or intergenic space with strict exon-only counting. A high "NoFeatures" count is expected. "MultiMapping" should be 0 because multi-mappers were excluded at the STAR step.

## 6. Prepare Count Matrix for R

The featureCounts output contains annotation columns (Chr, Start, End, Strand, Length) and a comment header. We strip these to produce a clean gene × sample matrix for DESeq2.

In [14]:
# Strip annotation columns; keep Geneid + count columns only
grep -v '^#' counts/all_samples.counts | cut -f1,7- > counts/_raw.tsv

# Build clean header
HEADER="Geneid"
for label in "${LABELS[@]}"; do HEADER="$HEADER\t$label"; done

# Replace the BAM-path header with short labels
tail -n +2 counts/_raw.tsv > counts/_body.tsv
printf "$HEADER\n" | cat - counts/_body.tsv > counts/count_matrix.tsv
rm counts/_raw.tsv counts/_body.tsv

echo "=== Count matrix preview ==="
head -5 counts/count_matrix.tsv | column -t
echo ""
echo "Matrix size:"
echo "  Genes: $(( $(wc -l < counts/count_matrix.tsv) - 1 ))"
echo "  Samples: $(( $(head -1 counts/count_matrix.tsv | awk '{print NF}') - 1 ))"

grep: counts/all_samples.counts: No such file or directory


=== Count matrix preview ===


Geneid  CN1  CN2  CN3  AD1  AD2  AD3


Matrix size:


  Genes: 0


  Samples: 6


In [15]:
# Sanity check: known blood and AD marker genes
echo "=== Key gene counts per sample ==="
printf "%-12s %6s %6s %6s %6s %6s %6s\n" "Gene" "CN1" "CN2" "CN3" "AD1" "AD2" "AD3"
for gene in HBB HBA1 MALAT1 TP53 CLEC5A MPO ELANE S100A8 S100A9 TREM2; do
    row=$(grep -P "^${gene}\t" counts/count_matrix.tsv)
    if [ -n "$row" ]; then
        printf "%-12s %6s %6s %6s %6s %6s %6s\n" \
            $(echo "$row" | awk '{print $1, $2, $3, $4, $5, $6, $7}')
    fi
done

=== Key gene counts per sample ===


Gene            CN1    CN2    CN3    AD1    AD2    AD3


## Critical Discussion

**Why biological replicates matter:**
The previous analysis used n=1 per group, which makes any fold-change estimate biologically meaningless — a single individual difference cannot be distinguished from group-level disease signal. DESeq2 uses per-gene dispersion estimates derived from within-group variance across replicates. Without replicates it borrows global dispersion priors, leading to unreliable p-values and inflated false discovery rates. Three replicates per group is the accepted minimum.

**Haemoglobin dominance in whole blood:**
HBB, HBA1, and HBA2 consistently top the count table, often comprising 30–50% of all assigned reads. This is biologically expected (erythrocytes lack nuclei but retain mRNA) but reduces effective sequencing depth for immune and neurological gene detection. Globin depletion during library preparation would improve sensitivity; its absence is a known limitation of this dataset.

**Strand specificity:**
The featureCounts `-s 0` (unstranded) setting is used because the Nakayama library preparation protocol is not strand-specific. Applying a wrong strandedness assumption can reduce assigned reads by >50%; when unknown, unstranded is the safer default.

**`--outFilterMultimapNmax 1` trade-off:**
Excluding multi-mappers prevents ambiguous gene assignments but discards ~10–15% of reads that map to repetitive or paralogous regions. Genes in large families (HLA, S100, olfactory receptors) are systematically under-counted. For the purpose of immune pathway analysis in AD, this is acceptable — the genes of primary interest (TREM2, CLU, CR1, BIN1) are singletons.

**Proceed to R notebook:** `counts/count_matrix.tsv` is ready for DESeq2 differential expression analysis.